In [0]:
%run /Workspace/Users/gustavosousa.md20@gmail.com/retreino_desloc_databricks/00_struct_table

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

import mlflow
mlflow.autolog(disable=True)

In [0]:
df = spark.table("tbl_ml").toPandas()
X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]


### Treinamento

In [0]:
colunas_minmax = [
    'dia', 'dia_semana', 'mes', 'hora', 'distancia','latitude_origem',
    'longitude_origem', 'latitude_destino', 'longitude_destino'
]

preprocessador = ColumnTransformer(
    transformers=[
        ('drop', 'drop', ['periodo']),
        ('minmax', MinMaxScaler(), colunas_minmax)
    ],
    remainder='passthrough'  # mantém as outras colunas sem alteração
)

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('knn', KNeighborsRegressor(n_neighbors=5))
])

In [0]:
experiment_name = "/Users/gustavosousa.md20@gmail.com/grc-preco-corrida"

try:
    experiment_id = mlflow.get_experiment_by_name(name=experiment_name).experiment_id
except:
    experiment_id = mlflow.create_experiment(name=experiment_name)

In [0]:
from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_schema = Schema([
    ColSpec("integer", "dia"),
    ColSpec("integer", "dia_semana"),
    ColSpec("integer", "mes"),
    ColSpec("integer", "hora"),
    ColSpec("integer", "periodo"),
    ColSpec("double", "distancia"),
    ColSpec("double", "latitude_origem"),
    ColSpec("double", "longitude_origem"),
    ColSpec("double", "latitude_destino"),
    ColSpec("double", "longitude_destino")
])

output_schema = Schema([ColSpec("double", "prediction")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(
    experiment_id=experiment_id
) as run:
    pipeline.fit(X, Y)
    mlflow.log_table(X.head(5), "train_sample.json")
    mlflow.sklearn.log_model(
        pipeline,
        "model",
        signature=signature
    )

In [0]:
with mlflow.start_run(
    experiment_id=experiment_id
) as run:
    pipeline.fit(X, Y)
    mlflow.log_table(X.head(5), "train_sample.json")
    mlflow.sklearn.log_model(
        pipeline,
        "model",
        input_example=X.head(1)
    )


In [0]:
model_uri = 'models:/m-982a0a4012ee45d29449cd481bbeb0d5'
model = mlflow.pyfunc.load_model(model_uri)
model.predict(X)


### Testando o Deployment

In [0]:
import json
import requests

In [0]:
TOKEN = ""

In [0]:
X.head(5)

In [0]:
post_data = json.dumps({"dataframe_split": X.head(10).to_dict(orient="split")})

In [0]:
r = requests.post(
     url = 'https://dbc-a7813a4b-db6a.cloud.databricks.com/serving-endpoints/knn/invocations',
    headers={
        'Authorization': f'Bearer {TOKEN}',
        'Content-Type': 'application/json'
    },
    data=post_data
)

In [0]:
print(r.json())